In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import pickle
import os
import glob

import pingouin as pg
from scipy import stats
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'


In [ ]:
# set up directories
basedir = '/path/to/pain-induced-manifold-changes'
datdir = os.path.join(basedir, 'data', 'manifold_geom_data_results')
savedir = '/dir/to/save/figure'

files = glob.glob(os.path.join(datdir, '*.pickle'))


In [ ]:
# load all files and concatenate them into a single dataframe
for i, file in enumerate(files):
   # print(file)
    if i==0:
        with open(file, 'rb') as f:
            df_all = pickle.load(f)
    else:
        with open(file, 'rb') as f:
            df_temp = pickle.load(f)
        df_all = pd.concat([df_all, df_temp], ignore_index=True)


In [ ]:
# calculate effective margin
df_all['Eff_margin'] = df_all['Radius']*np.sqrt(df_all['Dimensions'])

In [ ]:
# calculate mean and std for each subject, roi, and condition
df_all_mean = df_all.groupby(['Subject', 'ROI', 'Condition']).mean().reset_index()
df_all_std = df_all.groupby(['Subject', 'ROI', 'Condition']).std().reset_index()

df_all_long = pd.melt(df_all_mean, id_vars = ['Subject','ROI', 'Condition'], 
                      value_vars=['Capacity', 'Radius', 'Dimensions', 'Eff_margin'], 
                      var_name='ManifoldGeomMeasure', 
                      value_name='Value', 
                      ignore_index=False)
df_all_long = df_all_long.reset_index()


In [ ]:
# visualize the results using seaborn
save_figure = 0
cols = ["#fee0d2", "#fc9272", "#de2d26"]
rois = ['V1', 'V2', 'hV4', 'LO']
conditions = ['Warmth','LowHeat','HighHeat']

g = sns.FacetGrid(df_all_long, col='ManifoldGeomMeasure', sharey = False, hue= 'Condition', hue_order = conditions, palette=cols)
g.map(sns.pointplot, 'ROI', 'Value',order=rois) # errorbar defalut: CI 95% 
g.add_legend()


ManifoldGeomMeasure = ['Capacity', 'Radius', 'Dimensions', 'Effective margin'] 
ax_labels = [r'$\alpha_M$', r'$R_M$', r'$D_M$', 'a.u.'] 



for r in range(len(ax_labels)):
    g.axes[0][r].set_ylabel(f'{ax_labels[r]}', fontsize=12)
    g.axes[0][r].tick_params(labelsize=8)
    g.axes[0][r].set_title(f'{ManifoldGeomMeasure[r]}', fontsize=15)
    

if save_figure:
    plt.savefig(os.path.join(savedir, f'pain-induced-manifold-changes_manifold_geom_results.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ANOVA and post hoc


# ---------- shared formatting helpers ----------

def to_superscript(n):
    """Convert an integer to unicode superscript (e.g., -26 -> ⁻²⁶)."""
    sup = str.maketrans('0123456789-', '⁰¹²³⁴⁵⁶⁷⁸⁹⁻')
    return str(n).translate(sup)


def fmt_p_num(x):
    """p-value to three decimals; scientific notation (n.nnn × 10⁻ⁿ)
    when p < .001. Blank for NaN."""
    if pd.isna(x):
        return ''
    if x >= .001:
        return f'{x:.3f}'.lstrip('0')
    mantissa, exp = f'{x:.3e}'.split('e')
    return f'{mantissa} × 10{to_superscript(int(exp))}'



# ---------- ANOVA table ----------

def rm_anova_paper_table_combined(data, dv='mean_acc', subject='sub',
                                  within=('cond', 'roi'), labels=None):
    """pingouin two-way rm ANOVA -> publication-ready table.
    df/SS/MS are uncorrected (preserves additivity); ε and GG-corrected
    p-values are reported as additional columns."""
    aov = pg.rm_anova(data=data, dv=dv, within=list(within),
                      subject=subject, detailed=True)
    aov.columns = [c.replace('-', '_') for c in aov.columns]  # unify p-unc / p_unc

    if labels is None:
        labels = {'cond': 'Condition', 'roi': 'ROI',
                  'cond * roi': 'Condition × ROI'}

    # Subject and Total terms, computed directly from the raw data
    grand = data[dv].mean()
    ss_total = ((data[dv] - grand) ** 2).sum()
    df_total = len(data) - 1
    sub_means = data.groupby(subject)[dv].mean()
    n_cells = data.groupby(subject).size().iloc[0]  # cells per subject (12 here)
    ss_sub = (n_cells * (sub_means - grand) ** 2).sum()
    df_sub = sub_means.size - 1

    blank = dict(F=np.nan, p_unc=np.nan, eps=np.nan,
                 p_GG=np.nan, eta2G=np.nan)
    rows = [dict(Source='Subject', SS=ss_sub, df=df_sub,
                 MS=ss_sub / df_sub, **blank)]

    for _, r in aov.iterrows():
        lab = labels.get(r['Source'], r['Source'])
        ms_err = r['MS'] / r['F']      # recover error term from F = MS_effect / MS_error
        ss_err = ms_err * r['ddof2']
        rows.append(dict(Source=lab, SS=r['SS'], df=int(r['ddof1']),
                         MS=r['MS'], F=r['F'], p_unc=r['p_unc'],
                         eps=r['eps'], p_GG=r['p_GG_corr'],
                         eta2G=r['ng2']))
        rows.append(dict(Source=f'Error ({lab})', SS=ss_err,
                         df=int(r['ddof2']), MS=ms_err, **blank))

    rows.append(dict(Source='Total', SS=ss_total, df=df_total,
                     MS=np.nan, **blank))
    table = pd.DataFrame(rows)

    # Sanity check: both SS and df should sum to the Total row
    body = table[table['Source'] != 'Total']
    if not np.isclose(body['SS'].sum(), ss_total, rtol=1e-3):
        print(f"Warning: sum of SS ({body['SS'].sum():.4f}) "
              f"!= SS_total ({ss_total:.4f})")
    if body['df'].sum() != df_total:
        print(f"Warning: sum of df ({body['df'].sum():.0f}) "
              f"!= df_total ({df_total})")

    # Formatting for publication
    fmt = table.copy()
    fmt['SS'] = fmt['SS'].map(lambda x: f'{x:.3f}')
    fmt['df'] = fmt['df'].map(lambda x: f'{x:.0f}')
    fmt['MS'] = fmt['MS'].map(lambda x: '' if pd.isna(x) else f'{x:.4f}')
    fmt['F'] = fmt['F'].map(lambda x: '' if pd.isna(x) else f'{x:.2f}')
    fmt['p_unc'] = fmt['p_unc'].map(fmt_p_num)
    fmt['eps'] = fmt['eps'].map(lambda x: '' if pd.isna(x)
                                else f'{x:.3f}'.lstrip('0'))
    fmt['p_GG'] = fmt['p_GG'].map(fmt_p_num)
    fmt['eta2G'] = fmt['eta2G'].map(lambda x: '' if pd.isna(x)
                                    else f'{x:.3f}'.lstrip('0'))
    fmt = fmt.rename(columns={'p_unc': 'p', 'sig_unc': '',
                              'eps': 'ε', 'p_GG': 'p (GG)',
                              'sig_GG': ' ', 'eta2G': 'η²G'})
    # column order: p right before its flag column
    fmt = fmt[['Source', 'SS', 'df', 'MS', 'F', 'p',
               'ε', 'p (GG)', 'η²G']]
    return table, fmt


# ---------- Post hoc table ----------

def posthoc_paper_table(data, dv='mean_acc', subject='sub',
                        factor='cond', padjust='fdr_bh', labels=None):
    """Pairwise post hoc comparisons for one within-subject factor
    (paired t-tests with multiple-comparison correction),
    formatted as a publication-ready table."""
    # Collapse across the other factor: one value per subject x level
    agg = data.groupby([subject, factor])[dv].mean().reset_index()
    wide = agg.pivot(index=subject, columns=factor, values=dv)

    ph = pg.pairwise_tests(data=agg, dv=dv, within=factor,
                           subject=subject, padjust=padjust,
                           effsize='cohen')
    ph.columns = [c.replace('-', '_') for c in ph.columns] 
    # Mean difference (A - B) and paired 95% CI, per comparison
    stats_rows = []
    for _, r in ph.iterrows():
        d = wide[r['A']] - wide[r['B']]
        se = d.std(ddof=1) / np.sqrt(len(d))
        tcrit = stats.t.ppf(0.975, len(d) - 1)
        stats_rows.append((d.mean(), d.mean() - tcrit * se,
                           d.mean() + tcrit * se))
    ph[['mean_diff', 'ci_low', 'ci_high']] = pd.DataFrame(
        stats_rows, index=ph.index)

    # Optional relabeling (e.g., 0/1/2 -> condition names)
    if labels:
        ph['A'] = ph['A'].map(labels).fillna(ph['A'])
        ph['B'] = ph['B'].map(labels).fillna(ph['B'])

    fmt = pd.DataFrame({
        'Comparison': ph['A'].astype(str) + ' vs. ' + ph['B'].astype(str),
        'Mean diff.': ph['mean_diff'].map(lambda x: f'{x:.4f}'),
        '95% CI': ph.apply(lambda r: f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]",
                           axis=1),
        't': ph['T'].map(lambda x: f'{x:.2f}'),
        'df': ph['dof'].map(lambda x: f'{x:.0f}'),
        'p': ph['p_unc'].map(fmt_p_num),
        f'p ({padjust})': ph['p_corr'].map(fmt_p_num),
        "Cohen's d": ph['cohen'].map(lambda x: f'{x:.2f}'),
    })
    return ph, fmt

cond_labels = {'Warmth': 'Warmth', 'LowHeat': 'Low heat', 'HighHeat': 'High heat'}
var = 'Capacity' # select measure to test: 'Capacity', 'Radius', 'Dimensions', 'Eff_margin'

var_to_test_df = df_all_long[df_all_long['ManifoldGeomMeasure']==var]

raw_aov, aov_table = rm_anova_paper_table_combined(var_to_test_df, dv='Value', subject='Subject',
                                                   within=('Condition', 'ROI'),
                                                   labels = {'Condition': 'Heat level', 'ROI': 'ROI',
                                                             'Condition * ROI': 'Heat level × ROI'})
raw_cond, cond_table = posthoc_paper_table(var_to_test_df, dv='Value', subject='Subject',
                                           factor='Condition', padjust='fdr_bh', labels=cond_labels)
raw_roi, roi_table = posthoc_paper_table(var_to_test_df, dv='Value', subject='Subject',
                                         factor='ROI', padjust='fdr_bh', labels=cond_labels)

In [ ]:
print(f'ANOVA results for {dat_type} -- {var}')
aov_table.style.hide(axis='index')

In [ ]:
print(f'Post-hoc results for {dat_type} -- {var}')
cond_table.style.hide(axis='index')

In [ ]:
print(f'Post-hoc results for {dat_type} -- {var}')
roi_table.style.hide(axis='index')